# LatentDriver Visualization (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_visualize_colab.ipynb)

This notebook runs a small visualization job and surfaces the generated MP4 or PDF artifacts from the patched upstream simulator.


## What this notebook does

1. sync this repo
2. mount Drive and bind assets/results
3. clone and patch the pinned LatentDriver fork
4. optionally install the heavy runtime
5. run a visualization job on a smoke tier
6. display the generated artifact directly in Colab

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


$ git clone --branch main https://github.com/achyutmorang/latentdriver-waymax-experiments.git /content/latentdriver-waymax-experiments
Cloning into '/content/latentdriver-waymax-experiments'...
$ /usr/bin/python3 -m pip install -e /content/latentdriver-waymax-experiments
Obtaining file:///content/latentdriver-waymax-experiments
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for latentdriver-waymax-experiments (pyproject.toml): started
  Building editable for latentdriver-waymax-experiments (pyproject.toml): finished with stat

In [2]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


Mounted at /content/drive
{
  "checkpoints": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/checkpoints",
  "drive_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments",
  "preprocessed": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/preprocessed",
  "results": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs",
  "smoke": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/smoke"
}


In [3]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


$ /usr/bin/python3 scripts/bootstrap_upstream.py
Cloning into '/content/latentdriver-waymax-experiments/external/LatentDriver'...
Note: switching to '721bd6e1f87373457b743d0e0a9606d4d0727b6f'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 721bd6e Update README.md
{
  "fork_repo_url": "https://github.com/achyutmorang/LatentDriver.git",
  "patch_path": "/content/latentdriver-waymax-experiments/patches/latentdriver_eval_contract.patch",
  "patch_state": "applied",
  "pinned_commit": "721bd6e1f87373457b743d0e0a96

In [4]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


$ /usr/bin/python3 scripts/setup_colab_runtime.py --editable-project
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.8 MB/s eta 0:00:00
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Cloning https://github.com/waymo-research/waymax.git (to revision main) to /tmp/pip-install-j_bid41i/waymo-waymax_4fcd840832f7491eb930e236bff32edf
  Running command git clone --f

In [5]:
VIS_MODEL = "latentdriver_t2_j3"
VIS_TIER = "smoke_reactive"
VIS_SEED = 0
VIS_MODE = "video"  # switch to "image" for PDFs
print(json.dumps({
    "model": VIS_MODEL,
    "tier": VIS_TIER,
    "seed": VIS_SEED,
    "vis": VIS_MODE,
}, indent=2, sort_keys=True))


{
  "model": "latentdriver_t2_j3",
  "seed": 0,
  "tier": "smoke_reactive",
  "vis": "video"
}


In [6]:
import json

output = run([
    sys.executable,
    "scripts/run_visualization.py",
    "--model",
    VIS_MODEL,
    "--tier",
    VIS_TIER,
    "--seed",
    str(VIS_SEED),
    "--vis",
    VIS_MODE,
], cwd=REPO_DIR)
payload = json.loads(output)
print(json.dumps(payload, indent=2, sort_keys=True))


$ /usr/bin/python3 scripts/run_visualization.py --model latentdriver_t2_j3 --tier smoke_reactive --seed 0 --vis video
{
  "checkpoint_path": "/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J3.ckpt",
  "command": [
    "/usr/bin/python3",
    "simulate.py",
    "method=latentdriver",
    "++waymax_conf.path=/content/latentdriver-waymax-experiments/artifacts/assets/smoke/validation_smoke.tfrecord@1",
    "++data_conf.path_to_processed_map_route=/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/smoke/val_preprocessed_path",
    "++metric_conf.intention_label_path=/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/smoke/val_intention_label",
    "++batch_dims=[1,1]",
    "++ego_control_setting.npc_policy_type=idm",
    "++method.ckpt_path=/content/latentdriver-waymax-experiments/artifacts/assets/checkpoints/lantentdriver_t2_J3.ckpt",
    "++vis=video",
    "++run.seed=0",
    "++run.max_batches=1",
    "++run.

In [7]:
from IPython.display import Video, display

vis_root = Path(payload["vis_dir"])
mp4s = sorted(vis_root.glob("**/*.mp4"))
pdfs = sorted(vis_root.glob("**/*.pdf"))
print(json.dumps({
    "vis_root": str(vis_root),
    "mp4_count": len(mp4s),
    "pdf_count": len(pdfs),
}, indent=2, sort_keys=True))

if mp4s:
    print(f"Displaying {mp4s[0]}")
    display(Video(str(mp4s[0]), embed=True))
elif pdfs:
    print("Generated PDF files:")
    for path in pdfs[:10]:
        print(path)
else:
    print("No visualization files found. Inspect stderr.log in the run directory.")


{
  "mp4_count": 0,
  "pdf_count": 0,
  "vis_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs/20260410T184144Z_smoke_reactive_latentdriver_t2_j3_seed0/vis"
}
No visualization files found. Inspect stderr.log in the run directory.


## Next step

After you have one visualization artifact, move back to the public evaluation notebook and run the standardized suite on the full validation artifacts.